# Forecast Visualization — Model Evaluation & 7-Day Forecast

Interactive (Plotly) visualization of the **trained models** and the **daily 7-day forecasts**.

Sections:
1. **Model evaluation** — actual vs predicted on the held-out test set (last 20%, chronological)
2. **7-day forecast** — the latest predictions from `predictions/*.csv` (every model that has a saved forecast CSV)

The evaluation data is the shared **V3.1** feature table (V2.5 + grid `fi_*` + nuclear `nuclear_*`);
each model selects its own trained columns via its saved `feature_cols`.

**Which model is evaluated?** Whatever `MODEL_NAME` is set to in Section 1. Default = the **best
overall model: LightGBM V3.1** (`lightgbm_v3_1`, MAE 2.6390). Change it to `xgboost_v4`
(best XGBoost, MAE 2.7020) or any other saved model in `models/saved/` to compare.

Every figure renders interactively **directly in this notebook** — no image files are saved.

> Usage: run top to bottom. Re-run Section 2 after a new daily forecast
> (`python src/predict_system.py`) to refresh the charts.

## 0. Setup 


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── paths (this notebook lives in data_visualization/) ───────────────────
DATA_PATH   = Path('../data/convertData/V3.1_15min_features.csv')
MODELS_DIR  = Path('../models/saved')
PREDICTIONS = Path('../predictions')

HELSINKI = 'Europe/Helsinki'


def show(fig):
    """Display a figure interactively in the notebook."""
    fig.show()


## 1. Model Evaluation 


In [11]:
# Load the V3.1 feature table and normalise the time axis
df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert(HELSINKI)
df = df.sort_values('datetime').reset_index(drop=True)

# Chronological 80/20 split (no shuffle — time-series)
X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
train_end = n - int(n * 0.20)
X_test  = X.iloc[train_end:].reset_index(drop=True)
y_test  = y.iloc[train_end:].reset_index(drop=True)
test_dt = df['datetime'].iloc[train_end:].reset_index(drop=True)
print(f'Train rows: {train_end:,} | Test rows: {n - train_end:,}')


Train rows: 84,173 | Test rows: 21,043


In [12]:
# Every model available in models/saved/ (13 live models)
available = sorted(p.stem for p in MODELS_DIR.glob('*.pkl'))
print('Available models:', available)

# Load a saved model and predict on the test set.
# Best overall : lightgbm_v3_1 (MAE 2.6390)  <-- default
# Best XGBoost : xgboost_v4    (MAE 2.7020)
# Others       : xgboost_v2_5_3, xgboost_v3, xgboost_v2_5_2, lightgbm_v2_5_2, ...
MODEL_NAME = 'lightgbm_v3_1'
meta = joblib.load(MODELS_DIR / f'{MODEL_NAME}.pkl')
model = meta['model']
feature_cols = meta['feature_cols']

y_pred = model.predict(X_test[feature_cols])

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f'{MODEL_NAME}: MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')

Available models: ['lightgbm_v2', 'lightgbm_v2_5', 'lightgbm_v2_5_2', 'lightgbm_v3_1', 'xgboost_v1', 'xgboost_v1_5', 'xgboost_v2', 'xgboost_v2_5', 'xgboost_v2_5_2', 'xgboost_v2_5_3', 'xgboost_v3', 'xgboost_v4']
lightgbm_v3_1: MAE=2.6390 | RMSE=7.8957 | R2=0.9740


In [13]:
# 1.1 Actual vs Predicted - time series (first 7 days of the test set)
steps = 7 * 24 * 4
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dt[:steps], y=y_test[:steps], name='Actual', mode='lines'))
fig.add_trace(go.Scatter(x=test_dt[:steps], y=y_pred[:steps], name='Predicted', mode='lines',
                         line=dict(width=1.5)))
fig.update_layout(title=f'1.1 Actual vs Predicted - First 7 Days of Test ({MODEL_NAME})',
                  xaxis_title='Time', yaxis_title='Price (EUR/MWh)')
show(fig)


In [14]:
# 1.2 Actual vs Predicted - scatter with identity line (sampled for performance)
rng = np.random.RandomState(42)
idx = rng.choice(len(y_test), size=min(20000, len(y_test)), replace=False)

fig = px.scatter(x=y_test.values[idx], y=y_pred[idx], opacity=0.3,
                 title=f'1.2 Actual vs Predicted Scatter ({MODEL_NAME})',
                 labels={'x': 'Actual (EUR/MWh)', 'y': 'Predicted (EUR/MWh)'})

lims = [float(min(y_test.min(), y_pred.min())), float(max(y_test.max(), y_pred.max()))]
fig.add_trace(go.Scatter(x=lims, y=lims, mode='lines', name='y=x',
                         line=dict(color='red', dash='dash')))
show(fig)


In [15]:
# 1.3 Residual distribution (residual = actual - predicted)
residuals = y_test.values - y_pred
fig = px.histogram(residuals, nbins=80,
                   title=f'1.3 Residual Distribution ({MODEL_NAME})',
                   labels={'value': 'Residual (EUR/MWh)', 'count': 'Count'})
fig.add_vline(x=0, line_dash='dash', line_color='red')
show(fig)
print(f'Residual mean={residuals.mean():.4f} std={residuals.std():.4f}')


Residual mean=0.0762 std=7.8953


In [16]:
# 1.4 Feature importance (top 20)
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
fig = px.bar(imp.tail(20), orientation='h',
             title=f'1.4 Top 20 Feature Importances ({MODEL_NAME})',
             labels={'index': 'Feature', 'value': 'Importance'})
show(fig)


## 2. 7-Day Forecast 


In [17]:
def load_forecasts():
    """Load every <model>_forecasts.csv into a dict keyed by model name."""
    frames = {}
    for path in sorted(PREDICTIONS.glob('*_forecasts.csv')):
        f = pd.read_csv(path, parse_dates=['run_date', 'target_datetime'])
        name = path.stem.replace('_forecasts', '')
        if f['target_datetime'].dt.tz is None:
            f['target_datetime'] = f['target_datetime'].dt.tz_localize('UTC').dt.tz_convert(HELSINKI)
        else:
            f['target_datetime'] = f['target_datetime'].dt.tz_convert(HELSINKI)
        frames[name] = f
    return frames

forecasts = load_forecasts()
print('Loaded forecast files:', len(forecasts))
for name, f in forecasts.items():
    print(f'  {name}: {len(f)} rows | latest run {f["run_date"].max()}')


Loaded forecast files: 13
  lightgbm_v2_5_2: 6720 rows | latest run 2026-08-25 18:24:39.927952+03:00
  lightgbm_v2_5: 13440 rows | latest run 2026-08-25 18:24:39.927952+03:00
  lightgbm_v2: 3360 rows | latest run 2026-08-25 18:24:39.927952+03:00
  lightgbm_v3_1: 672 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v1_5: 13440 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v1: 3360 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v2_5_2: 9408 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v2_5_3: 6720 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v2_5: 13440 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v2: 3360 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v3_1_enh: 672 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v3: 672 rows | latest run 2026-08-25 18:24:39.927952+03:00
  xgboost_v4: 672 rows | latest run 2026-08-25 18:24:39.927952+03:00


In [20]:
# 2.1 Latest 7-day forecast for every model on one chart
fig = go.Figure()
for name, f in forecasts.items():
    f_latest = f[f['run_date'] == f['run_date'].max()]
    fig.add_trace(go.Scatter(x=f_latest['target_datetime'], y=f_latest['predicted_price'],
                             name=name, mode='lines'))
fig.update_layout(title='2.1 7-Day Forecast - All Models ',
                  xaxis_title='Time', yaxis_title='Predicted Price (EUR/MWh)')
show(fig)
